# ForecastWise — MR D.I.Y. | Member 3: Forecasting & Accuracy Analyst
**GFB3213 / GEB3213 Forecasting Methods for Management — September 2026**

**Core question:** Which numerical method forecasts MR D.I.Y. quarterly revenue more reliably?

| Item | Detail |
|---|---|
| Input (from Member 2) | `MRDIY_Cleaned_Revenue.csv`: 18 consecutive standalone quarters, 2022-Q1 to 2026-Q2, RM million |
| Method A | 4-Quarter Moving Average (n = 4) |
| Method B | Least Squares Trend Projection (Ŷ = a + bt), refitted at every forecast origin |
| Validation | Rolling 1-step-ahead forecasts, **2025-Q3 to 2026-Q2** (4 quarters), the same quarters for both methods |
| Robustness check | Same procedure on the longest window both methods can cover: **2023-Q1 to 2026-Q2** (14 quarters) |
| Accuracy measures | MAD, MSE, MAPE (all actuals > 0, so MAPE is valid) plus bias (mean error) as a diagnostic |
| Output (to Member 5) | `src/forecasting.py`, `src/accuracy.py`, validation tables, selected method, next-quarter forecast, charts |

**How to run in Colab:** *Runtime → Run all*. If `MRDIY_Cleaned_Revenue.csv` is not already in the Colab file panel, Cell 2 asks you to upload it.

## Step 1 — Setup and load Member 2's cleaned dataset

In [ ]:
# Step 1A: Setup
import os, sys, shutil, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("src", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# Chart style (shared by every figure in this notebook)
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e5e4e0"
C_MA, C_LS = "#2a78d6", "#eb6834"      # blue = Moving Average, orange = Least Squares
plt.rcParams.update({
    "font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
})
print("Setup complete.")

In [ ]:
# Step 1B: Load the cleaned revenue series from Member 2
DATA_FILE = "MRDIY_Cleaned_Revenue.csv"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print(f"'{DATA_FILE}' not found - please upload Member 2's file.")
        uploaded = files.upload()
        DATA_FILE = next(iter(uploaded))      # Colab may rename it, e.g. '... (1).csv'
    except ImportError:
        raise FileNotFoundError(f"Place {DATA_FILE} next to this notebook.")

df = pd.read_csv(DATA_FILE)

# Re-check Member 2's hand-off before forecasting (no silent assumptions)
q = pd.PeriodIndex(df["period"], freq="Q-DEC")
expected = pd.period_range(q.min(), q.max(), freq="Q-DEC")
assert list(df.columns) == ["period", "revenue_rm_million"], "Unexpected columns"
assert df["revenue_rm_million"].notna().all(), "Missing revenue values"
assert (df["revenue_rm_million"] > 0).all(), "Zero/negative revenue - MAPE would be invalid"
assert q.is_monotonic_increasing and len(q) == len(expected), "Quarters not consecutive"
assert len(df) >= 12, "Brief requires at least 12 consecutive periods"

df["t"] = np.arange(1, len(df) + 1)    # time index used by the least squares trend
print(f"Loaded {len(df)} consecutive quarters: {df.period.iloc[0]} to {df.period.iloc[-1]} (RM million)")
display(df)

## Step 2 — The two forecasting methods

**Method A: 4-Quarter Moving Average**

$$F_{t} = \frac{A_{t-1} + A_{t-2} + A_{t-3} + A_{t-4}}{4}$$

The forecast for quarter *t* is the average of the four most recent actual quarters. Using **n = 4** means every average spans one Q1, Q2, Q3 and Q4, which smooths out MR D.I.Y.'s quarter-to-quarter (seasonal) swings.

**Method B: Least Squares Trend Projection**

$$\hat{Y} = a + bt, \qquad b = \frac{\sum ty - n\,\bar{t}\,\bar{y}}{\sum t^{2} - n\,\bar{t}^{2}}, \qquad a = \bar{y} - b\,\bar{t}$$

*t* = 1 for 2022-Q1, 2 for 2022-Q2, … 18 for 2026-Q2; *y* = revenue (RM million); *b* = average change in revenue per quarter.
To avoid **future leakage**, *a* and *b* are re-estimated at every forecast origin using **only the quarters available at that time**.

Both methods are written once as reusable functions (next two cells). The same files go to Member 5 for the Streamlit app.

In [ ]:
%%writefile src/forecasting.py
"""
forecasting.py - ForecastWise MR D.I.Y. (Member 3)
Two taught forecasting methods + rolling one-step-ahead validation.
All revenue values are in RM million. No function uses data after its forecast origin.
"""
import numpy as np
import pandas as pd

MA = "4-Quarter Moving Average"
LS = "Least Squares Trend"


def moving_average_forecast(history, n=4):
    """F(t) = mean of the n most recent actual values in `history`."""
    h = np.asarray(history, dtype=float)
    if n < 1 or len(h) < n:
        raise ValueError(f"Moving average needs at least n={n} observations, got {len(h)}.")
    return float(h[-n:].mean())


def least_squares_fit(history):
    """Fit y = a + b*t with t = 1..len(history). Returns (a, b)."""
    y = np.asarray(history, dtype=float)
    if len(y) < 2:
        raise ValueError("Least squares trend needs at least 2 observations.")
    t = np.arange(1, len(y) + 1, dtype=float)
    n = len(y)
    b = ((t * y).sum() - n * t.mean() * y.mean()) / ((t ** 2).sum() - n * t.mean() ** 2)
    a = y.mean() - b * t.mean()
    return float(a), float(b)


def least_squares_forecast(history, steps_ahead=1):
    """Project the fitted trend to t = len(history) + steps_ahead."""
    a, b = least_squares_fit(history)
    return float(a + b * (len(history) + steps_ahead))


def forecast_next(history, method, n=4):
    """One-step-ahead forecast with the named method (MA or LS)."""
    if method == MA:
        return moving_average_forecast(history, n)
    if method == LS:
        return least_squares_forecast(history)
    raise ValueError(f"Unknown method: {method}")


def rolling_one_step_forecasts(df, start_period, n=4,
                               period_col="period", value_col="revenue_rm_million"):
    """
    For every quarter from `start_period` to the end of `df`, forecast it using
    ONLY the quarters before it (expanding window), then compare with the actual.
    Both methods are evaluated on exactly the same quarters.
    """
    periods = df[period_col].tolist()
    values = df[value_col].to_numpy(dtype=float)
    if start_period not in periods:
        raise ValueError(f"{start_period} not in data.")
    start = periods.index(start_period)
    if start < max(n, 2):
        raise ValueError(f"Need at least {max(n, 2)} quarters before {start_period}.")

    rows = []
    for i in range(start, len(values)):
        hist = values[:i]                       # data available at the forecast origin
        a, b = least_squares_fit(hist)
        ma_f = moving_average_forecast(hist, n)
        ls_f = a + b * (i + 1)                  # t of the target quarter = i + 1
        rows.append({
            "period": periods[i], "t": i + 1, "actual": values[i],
            "ma_forecast": ma_f, "ls_forecast": ls_f, "ls_a": a, "ls_b": b,
            "ma_error": values[i] - ma_f, "ls_error": values[i] - ls_f,
        })
    return pd.DataFrame(rows)


def next_period_label(period):
    """'2026-Q2' -> '2026-Q3'; '2026-Q4' -> '2027-Q1'."""
    year, qtr = period.split("-Q")
    year, qtr = int(year), int(qtr)
    return f"{year + 1}-Q1" if qtr == 4 else f"{year}-Q{qtr + 1}"


def apply_scenario(base_forecast, pct_change):
    """Hypothetical managerial adjustment, e.g. -5 (slowdown) or +10 (promotion)."""
    return float(base_forecast * (1 + pct_change / 100.0))

In [ ]:
%%writefile src/accuracy.py
"""
accuracy.py - ForecastWise MR D.I.Y. (Member 3)
Forecast error measures. Error convention: e = Actual - Forecast
(positive e = under-forecast, negative e = over-forecast).
"""
import numpy as np
import pandas as pd


def _arrays(actual, forecast):
    a = np.asarray(actual, dtype=float)
    f = np.asarray(forecast, dtype=float)
    if a.shape != f.shape or a.size == 0:
        raise ValueError("Actual and forecast must be non-empty and the same length.")
    return a, f


def mad(actual, forecast):
    """Mean Absolute Deviation = sum|A - F| / n   (RM million)."""
    a, f = _arrays(actual, forecast)
    return float(np.mean(np.abs(a - f)))


def mse(actual, forecast):
    """Mean Squared Error = sum(A - F)^2 / n   (RM million squared)."""
    a, f = _arrays(actual, forecast)
    return float(np.mean((a - f) ** 2))


def rmse(actual, forecast):
    """Root MSE - same units as revenue, used only as an error band."""
    return float(np.sqrt(mse(actual, forecast)))


def mape(actual, forecast):
    """Mean Absolute Percentage Error = (100/n) * sum|A - F|/A   (%). Invalid if any A = 0."""
    a, f = _arrays(actual, forecast)
    if np.any(a == 0):
        raise ValueError("MAPE is undefined when an actual value is zero.")
    return float(np.mean(np.abs((a - f) / a)) * 100)


def bias(actual, forecast):
    """Mean error (A - F). Positive = method tends to under-forecast."""
    a, f = _arrays(actual, forecast)
    return float(np.mean(a - f))


def accuracy_table(actual, forecasts):
    """forecasts: {method_name: forecast_array}. Returns one row per method."""
    rows = []
    for name, f in forecasts.items():
        rows.append({"Method": name, "MAD": mad(actual, f), "MSE": mse(actual, f),
                     "MAPE (%)": mape(actual, f), "Bias (mean error)": bias(actual, f)})
    return pd.DataFrame(rows).set_index("Method")


def select_method(primary, robustness=None, decision_metrics=("MAD", "MSE")):
    """
    Transparent selection rule:
      1. On the primary validation quarters, the method with the lower value on
         EVERY decision metric wins.
      2. If the metrics disagree, the method that wins every decision metric on the
         longer robustness backtest is chosen.
      3. If still unresolved, the lower primary MSE wins (it penalises large misses).
    Returns (method_name, reason).
    """
    def unanimous(tbl):
        winners = {tbl[m].idxmin() for m in decision_metrics}
        return winners.pop() if len(winners) == 1 else None

    w = unanimous(primary)
    if w:
        return w, f"Lowest {' and '.join(decision_metrics)} on the validation quarters."
    if robustness is not None:
        w = unanimous(robustness)
        if w:
            return w, (f"Validation metrics were split, so the longer backtest decided: "
                       f"{w} has the lowest {' and '.join(decision_metrics)} there.")
    w = primary["MSE"].idxmin()
    return w, "Metrics split on both windows; lower validation MSE used as tie-breaker."

In [ ]:
# Import Member 3's modules (reload so edits to the files are picked up)
sys.path.insert(0, os.path.abspath("src"))
import forecasting as fc, accuracy as acc
importlib.reload(fc); importlib.reload(acc)
print("Loaded:", fc.__file__, "|", acc.__file__)

## Step 3 — Calculation audit: test the functions before trusting them
Each function is checked against (a) a hand calculation on a tiny series, (b) NumPy's independent least squares solver, and (c) a direct hand calculation on the real data.

In [ ]:
# Step 3: Unit tests (every line must pass; any failure stops the notebook)
toy = [10, 20, 30, 40, 50]

# (a) Hand-checkable cases
assert fc.moving_average_forecast(toy, 4) == (20 + 30 + 40 + 50) / 4 == 35
a, b = fc.least_squares_fit(toy)                     # perfect line y = 10t
assert abs(a - 0) < 1e-9 and abs(b - 10) < 1e-9
assert abs(fc.least_squares_forecast(toy) - 60) < 1e-9
assert acc.mad([100, 200], [90, 210]) == 10
assert acc.mse([100, 200], [90, 210]) == 100
assert abs(acc.mape([100, 200], [90, 210]) - 7.5) < 1e-9
assert acc.bias([100, 200], [90, 190]) == 10
assert fc.next_period_label("2026-Q2") == "2026-Q3" and fc.next_period_label("2026-Q4") == "2027-Q1"
assert abs(fc.apply_scenario(1000, -5) - 950) < 1e-9 and abs(fc.apply_scenario(1000, 10) - 1100) < 1e-9

# Error handling
for bad in (lambda: fc.moving_average_forecast([1, 2, 3], 4),
            lambda: fc.least_squares_fit([5]),
            lambda: acc.mape([0, 10], [1, 10])):
    try:
        bad(); raise AssertionError("expected ValueError")
    except ValueError:
        pass

# (b) Least squares matches NumPy's independent solver on the real data
y = df["revenue_rm_million"].to_numpy()
b_np, a_np = np.polyfit(np.arange(1, len(y) + 1), y, 1)
a_fc, b_fc = fc.least_squares_fit(y)
assert abs(a_fc - a_np) < 1e-8 and abs(b_fc - b_np) < 1e-8

# (c) No look-ahead: the rolling forecast for each quarter equals a fresh fit on earlier data only
chk = fc.rolling_one_step_forecasts(df, "2025-Q3")
for _, r in chk.iterrows():
    hist = y[: int(r.t) - 1]
    assert abs(r.ma_forecast - hist[-4:].mean()) < 1e-9
    assert abs(r.ls_forecast - fc.least_squares_forecast(hist)) < 1e-9

print("All calculation-audit tests passed.")

## Step 4 — Rolling one-quarter-ahead validation (2025-Q3 to 2026-Q2)
For each validation quarter the methods see only the quarters before it. Example: to forecast 2025-Q4, the moving average uses 2025-Q1 to 2025-Q3 plus 2024-Q4, and the trend line is refitted on 2022-Q1 to 2025-Q3 (t = 1 to 15). This matches how a manager would actually forecast in real time.

In [ ]:
# Step 4: Rolling one-step-ahead forecasts on the validation quarters
VALIDATION_START = "2025-Q3"
val = fc.rolling_one_step_forecasts(df, VALIDATION_START)

val_view = val.rename(columns={
    "period": "Quarter", "actual": "Actual", "ma_forecast": "MA forecast",
    "ls_forecast": "LS forecast", "ls_a": "LS a", "ls_b": "LS b",
    "ma_error": "MA error (A-F)", "ls_error": "LS error (A-F)"}).drop(columns="t")
print("ROLLING 1-STEP-AHEAD FORECASTS - VALIDATION QUARTERS (RM million)")
display(val_view.round(3))
val_view.round(3).to_csv("outputs/MRDIY_M3_Validation_Forecasts.csv", index=False)

## Step 5 — Worked example (for the report and poster): forecasting 2026-Q2

In [ ]:
# Step 5: Worked example - both methods forecasting 2026-Q2 (t = 18)
target = "2026-Q2"
i = df.index[df.period == target][0]
hist = df.iloc[:i]                      # 2022-Q1 .. 2026-Q1 (17 quarters)
actual = df.revenue_rm_million.iloc[i]

print(f"WORKED EXAMPLE - forecasting {target} (t = {i + 1}); actual = RM {actual:,.3f} m\n")

# Method A
last4 = hist.tail(4)
ma_f = last4.revenue_rm_million.mean()
print("METHOD A: 4-Quarter Moving Average")
print("  F(2026-Q2) = (" + " + ".join(f"{v:,.3f}" for v in last4.revenue_rm_million)
      + ") / 4")
print(f"             = {last4.revenue_rm_million.sum():,.3f} / 4 = RM {ma_f:,.3f} m")
print(f"  Error = {actual:,.3f} - {ma_f:,.3f} = {actual - ma_f:+,.3f}  |  "
      f"APE = {abs(actual - ma_f) / actual * 100:.2f}%\n")

# Method B
t = hist.t.to_numpy(dtype=float); yv = hist.revenue_rm_million.to_numpy()
n = len(t); St, Sy, Sty, St2 = t.sum(), yv.sum(), (t * yv).sum(), (t ** 2).sum()
b = (Sty - n * t.mean() * yv.mean()) / (St2 - n * t.mean() ** 2)
a = yv.mean() - b * t.mean()
ls_f = a + b * (i + 1)
print("METHOD B: Least Squares Trend (fitted on t = 1..17 only)")
print(f"  n = {n},  sum t = {St:,.0f},  sum y = {Sy:,.3f},  sum ty = {Sty:,.3f},  sum t^2 = {St2:,.0f}")
print(f"  t-bar = {t.mean():.3f},  y-bar = {yv.mean():,.3f}")
print(f"  b = ({Sty:,.3f} - {n} x {t.mean():.3f} x {yv.mean():,.3f}) / ({St2:,.0f} - {n} x {t.mean():.3f}^2)"
      f" = {b:,.3f}  (RM m per quarter)")
print(f"  a = {yv.mean():,.3f} - {b:,.3f} x {t.mean():.3f} = {a:,.3f}")
print(f"  F(2026-Q2) = {a:,.3f} + {b:,.3f} x {i + 1} = RM {ls_f:,.3f} m")
print(f"  Error = {actual:,.3f} - {ls_f:,.3f} = {actual - ls_f:+,.3f}  |  "
      f"APE = {abs(actual - ls_f) / actual * 100:.2f}%")

# Cross-check against the reusable functions
assert abs(ma_f - val.set_index("period").loc[target, "ma_forecast"]) < 1e-9
assert abs(ls_f - val.set_index("period").loc[target, "ls_forecast"]) < 1e-9
print("\nWorked example matches the function output.")

## Step 6 — Accuracy comparison on identical quarters

In [ ]:
# Step 6A: Primary accuracy comparison - validation quarters 2025-Q3 to 2026-Q2
primary = acc.accuracy_table(val.actual, {fc.MA: val.ma_forecast, fc.LS: val.ls_forecast})
print(f"ACCURACY - VALIDATION QUARTERS {val.period.iloc[0]} to {val.period.iloc[-1]} (n = {len(val)})")
display(primary.round(3))

# Step 6B: Robustness check - longest window both methods can cover (MA needs 4 prior quarters)
ROBUST_START = "2023-Q1"
rob = fc.rolling_one_step_forecasts(df, ROBUST_START)
robust = acc.accuracy_table(rob.actual, {fc.MA: rob.ma_forecast, fc.LS: rob.ls_forecast})
print(f"\nACCURACY - ROBUSTNESS BACKTEST {rob.period.iloc[0]} to {rob.period.iloc[-1]} (n = {len(rob)})")
display(robust.round(3))

# Save both, in one tidy file
both = pd.concat({f"Validation {val.period.iloc[0]} to {val.period.iloc[-1]} (n={len(val)})": primary,
                  f"Backtest {rob.period.iloc[0]} to {rob.period.iloc[-1]} (n={len(rob)})": robust},
                 names=["Window", "Method"]).round(3)
both.to_csv("outputs/MRDIY_M3_Accuracy_Comparison.csv")
rob.round(3).to_csv("outputs/MRDIY_M3_Backtest_2023Q1_2026Q2.csv", index=False)

# Which method is better on each metric?
for name, tbl in (("Validation", primary), ("Backtest", robust)):
    print(f"{name:>10}: " + " | ".join(f"{m}: {tbl[m].idxmin()}" for m in ["MAD", "MSE", "MAPE (%)"]))

In [ ]:
# Step 6C: Method selection (rule written in accuracy.select_method)
SELECTED, REASON = acc.select_method(primary, robust, decision_metrics=("MAD", "MSE"))
print("SELECTED METHOD:", SELECTED)
print("REASON:", REASON)
print(f"\nBias check (mean error A-F, validation): MA = {primary.loc[fc.MA, 'Bias (mean error)']:+.2f}, "
      f"LS = {primary.loc[fc.LS, 'Bias (mean error)']:+.2f} RM m")
print(f"Bias check (mean error A-F, backtest):   MA = {robust.loc[fc.MA, 'Bias (mean error)']:+.2f}, "
      f"LS = {robust.loc[fc.LS, 'Bias (mean error)']:+.2f} RM m")
under = (rob.ma_error > 0).sum()
print(f"MA under-forecast in {under} of {len(rob)} backtest quarters (it lags a rising series).")

**Why MAD and MSE decide, not MAPE as a third vote:** the team plan names MAD and MSE as the required pair (MAPE optional). Revenue varies only between RM 905 m and RM 1,374 m, so MAPE is close to MAD divided by a near-constant level. Counting it as a separate vote would count the same information twice. MAPE is still reported for interpretation.

## Step 7 — Charts for the report, poster and video

In [ ]:
# Chart M3-1: Actual revenue vs rolling one-step-ahead forecasts
fig, ax = plt.subplots(figsize=(12, 5.2))
x_all = np.arange(len(df)); idx = {p: k for k, p in enumerate(df.period)}
xr = [idx[p] for p in rob.period]

v0, v1 = idx[val.period.iloc[0]], idx[val.period.iloc[-1]]
ax.axvspan(v0 - 0.5, v1 + 0.5, color="#f1f0ec", zorder=0)
ax.text((v0 + v1) / 2, 905, "Validation\nholdout", ha="center", va="bottom", color=INK_2, fontsize=9)

ax.plot(x_all, df.revenue_rm_million, color=INK, lw=2, marker="o", ms=5, label="Actual revenue", zorder=4)
ax.plot(xr, rob.ma_forecast, color=C_MA, lw=2, marker="s", ms=5, label=fc.MA, zorder=3)
ax.plot(xr, rob.ls_forecast, color=C_LS, lw=2, marker="^", ms=6, label=fc.LS, zorder=3)

# Direct labels at the right edge, nudged apart so they never overlap
labels = sorted([(df.revenue_rm_million.iloc[-1], "Actual", INK),
                 (rob.ma_forecast.iloc[-1], "Moving Avg", C_MA),
                 (rob.ls_forecast.iloc[-1], "LS Trend", C_LS)])
MIN_GAP = 28                                   # RM million between label centres
placed = []
for yv, txt, col in labels:
    y_lab = max(yv, placed[-1] + MIN_GAP) if placed else yv
    placed.append(y_lab)
    ax.annotate(txt, (len(df) - 1, yv), xytext=(len(df) - 0.6, y_lab), textcoords="data",
                va="center", color=INK, fontsize=9,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=col, lw=1))

ax.set_xticks(x_all, df.period, rotation=45)
ax.set_xlim(-0.5, len(df) + 1.0)
ax.set_ylabel("Revenue (RM million)")
ax.set_title("MR D.I.Y. Revenue: Actual vs Rolling One-Quarter-Ahead Forecasts",
             fontsize=13, fontweight="bold", loc="left", pad=12)
ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(loc="upper left", frameon=False)
plt.tight_layout()
plt.savefig("outputs/MRDIY_M3_Chart1_Forecast_vs_Actual.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Chart M3-2: Absolute forecast error per validation quarter
fig, ax = plt.subplots(figsize=(9, 4.8))
xv = np.arange(len(val)); w = 0.38
for off, col, err, name in ((-w / 2, C_MA, val.ma_error, fc.MA), (w / 2, C_LS, val.ls_error, fc.LS)):
    bars = ax.bar(xv + off, err.abs(), width=w - 0.03, color=col, label=name)
    for bar, e in zip(bars, err):
        ax.annotate(f"{abs(e):.1f}\n({'under' if e > 0 else 'over'})",
                    (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom",
                    fontsize=8, color=INK_2)
ax.set_xticks(xv, val.period)
ax.set_ylabel("Absolute error |A - F| (RM million)")
ax.set_ylim(0, max(val.ma_error.abs().max(), val.ls_error.abs().max()) * 1.25)
ax.set_title("Absolute Forecast Error by Validation Quarter", fontsize=13, fontweight="bold", loc="left", pad=12)
ax.grid(axis="y", color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.savefig("outputs/MRDIY_M3_Chart2_Absolute_Errors.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Chart M3-3: Poster-ready accuracy comparison table (both windows)
def fmt_rows(tbl):
    out = []
    for m, f in (("MAD (RM m)", "{:,.2f}"), ("MSE (RM m²)", "{:,.0f}"), ("MAPE (%)", "{:.2f}")):
        key = m.split(" ")[0] if m != "MAPE (%)" else "MAPE (%)"
        ma_v, ls_v = tbl.loc[fc.MA, key], tbl.loc[fc.LS, key]
        out.append([m, f.format(ma_v), f.format(ls_v), "LS" if ls_v < ma_v else "MA"])
    return out

hdr = ["Measure", "MA (n=4)", "LS Trend", "Lower"]
fig, axes = plt.subplots(1, 2, figsize=(11, 2.4))
for axx, tbl, ttl in ((axes[0], primary, f"Validation {val.period.iloc[0]}–{val.period.iloc[-1]} (4 qtrs)"),
                      (axes[1], robust, f"Backtest {rob.period.iloc[0]}–{rob.period.iloc[-1]} (14 qtrs)")):
    axx.axis("off")
    t_ = axx.table(cellText=fmt_rows(tbl), colLabels=hdr, cellLoc="center",
                   colWidths=[0.32, 0.24, 0.24, 0.2], bbox=[0, 0, 1, 0.82])
    t_.auto_set_font_size(False); t_.set_fontsize(10)
    for (r, c), cell in t_.get_celld().items():
        cell.set_edgecolor(GRID)
        if r == 0:
            cell.set_text_props(weight="bold", color=INK); cell.set_facecolor("#f1f0ec")
    axx.set_title(ttl, fontsize=11, fontweight="bold", color=INK, pad=4)
fig.suptitle(f"Forecast Accuracy Comparison  |  Selected: {SELECTED}", fontsize=12, fontweight="bold", y=1.08)
plt.savefig("outputs/MRDIY_M3_Chart3_Accuracy_Table.png", dpi=300, bbox_inches="tight")
plt.show()

## Step 8 — Next-quarter forecast (2026-Q3) and scenario hook for Members 4 and 5
Both methods are now fitted on all 18 quarters. The ±5% / +10% adjustments are the **illustrative managerial scenarios from the team plan**. They are hypothetical adjustments, not statistical predictions; Member 4 owns the final assumptions.

In [ ]:
# Step 8: Forecast 2026-Q3 with all available data
y_all = df.revenue_rm_million.to_numpy()
next_q = fc.next_period_label(df.period.iloc[-1])
a_all, b_all = fc.least_squares_fit(y_all)
fcst = {fc.MA: fc.forecast_next(y_all, fc.MA), fc.LS: fc.forecast_next(y_all, fc.LS)}
band = acc.rmse(val.actual, val.ls_forecast if SELECTED == fc.LS else val.ma_forecast)

print(f"NEXT-QUARTER FORECAST: {next_q} (t = {len(y_all) + 1})")
print(f"  {fc.MA}: ({' + '.join(f'{v:,.3f}' for v in y_all[-4:])}) / 4 = RM {fcst[fc.MA]:,.3f} m")
print(f"  {fc.LS}: {a_all:,.3f} + {b_all:,.3f} x {len(y_all) + 1} = RM {fcst[fc.LS]:,.3f} m")
print(f"  Selected method ({SELECTED}): RM {fcst[SELECTED]:,.3f} m  "
      f"(typical error ± RM {band:,.1f} m = validation RMSE)")

base = fcst[SELECTED]
scen = pd.DataFrame([
    {"Scenario": "Baseline (selected method)", "Adjustment (%)": 0},
    {"Scenario": "Consumer slowdown (illustrative)", "Adjustment (%)": -5},
    {"Scenario": "Promotional uplift (illustrative)", "Adjustment (%)": 10},
])
scen["Forecast (RM m)"] = [fc.apply_scenario(base, p) for p in scen["Adjustment (%)"]]
scen["Change vs baseline (RM m)"] = scen["Forecast (RM m)"] - base
scen.insert(0, "Quarter", next_q); scen.insert(1, "Method", SELECTED)
display(scen.round(3))

summary = pd.DataFrame([{"Quarter": next_q, "Method": m, "Forecast (RM m)": v,
                         "Selected": m == SELECTED} for m, v in fcst.items()])
summary.round(3).to_csv("outputs/MRDIY_M3_Next_Quarter_Forecast.csv", index=False)
scen.round(3).to_csv("outputs/MRDIY_M3_Scenario_Hook.csv", index=False)

## Step 9 — Report-ready text (Sections 5 and 7), poster strip and video notes

In [ ]:
# Step 9: Auto-generated text - numbers update automatically if the data change
P, R = primary, robust
vq = f"{val.period.iloc[0]} to {val.period.iloc[-1]}"; rq = f"{rob.period.iloc[0]} to {rob.period.iloc[-1]}"
ma_v, ls_v = P.loc[fc.MA], P.loc[fc.LS]; ma_r, ls_r = R.loc[fc.MA], R.loc[fc.LS]
worst = val.loc[val.ma_error.abs().idxmax()]

text = f'''
SECTION 5 - QUANTITATIVE FORECASTING METHODS (draft)
Two taught numerical methods were applied to MR D.I.Y.'s consolidated standalone quarterly revenue
(RM million, {df.period.iloc[0]} to {df.period.iloc[-1]}, {len(df)} quarters). Method A, the 4-quarter moving
average, forecasts each quarter as the mean of the four preceding actual quarters; n = 4 was chosen so that
every average contains one of each quarter and seasonal swings are averaged out. Method B, least squares trend
projection, fits Y = a + bt (t = 1 for {df.period.iloc[0]}). Fitted on all {len(df)} quarters, the trend is
Y = {a_all:,.3f} + {b_all:,.3f}t, meaning revenue has grown by about RM {b_all:,.1f} million per quarter on
average. To prevent look-ahead bias, both methods were evaluated with rolling one-quarter-ahead forecasts:
each forecast uses only the quarters available before it, and the trend line is refitted at every origin.
Worked example ({target}): the moving average gives RM {ma_f:,.3f} m and the trend line gives
RM {ls_f:,.3f} m against an actual of RM {actual:,.3f} m.

SECTION 7 - FORECAST ACCURACY AND METHOD SELECTION (draft)
On the validation quarters {vq} (n = {len(val)}), the moving average recorded MAD = {ma_v.MAD:,.2f},
MSE = {ma_v.MSE:,.0f} and MAPE = {ma_v['MAPE (%)']:.2f}%, against the trend's MAD = {ls_v.MAD:,.2f},
MSE = {ls_v.MSE:,.0f} and MAPE = {ls_v['MAPE (%)']:.2f}%. The measures disagree: the moving average has the
lower average error, but the trend has the lower MSE because the moving average produced the single largest
miss (under-forecasting {worst.period} by RM {worst.ma_error:,.1f} m). With only four validation quarters, the
same procedure was repeated over the longest window both methods can cover, {rq} (n = {len(rob)}). There, the
trend is better on every measure (MAD {ls_r.MAD:,.2f} vs {ma_r.MAD:,.2f}; MSE {ls_r.MSE:,.0f} vs
{ma_r.MSE:,.0f}; MAPE {ls_r['MAPE (%)']:.2f}% vs {ma_r['MAPE (%)']:.2f}%). The moving average also shows
systematic bias: its mean error is +{ma_r['Bias (mean error)']:,.1f} m (it under-forecast in
{(rob.ma_error > 0).sum()} of {len(rob)} quarters) because it lags behind a rising series, whereas the trend's
bias is {ls_r['Bias (mean error)']:+,.1f} m. {SELECTED} is therefore selected. For MR D.I.Y., persistent
under-forecasting risks under-stocking and under-staffing in a growing business, and MSE's penalty on large
misses matches the cost of a bad quarter. Limitations: the validation window is short; neither method models
seasonality explicitly (revenue dipped from Q2 to Q3 and rebounded in Q4 in every year 2022-2025); a straight-line trend assumes growth continues at a
constant rate and will not anticipate shocks; the results depend on the accuracy of the published figures.

NEXT QUARTER ({next_q}): {SELECTED} forecasts RM {fcst[SELECTED]:,.1f} m (typical error +/- RM {band:,.0f} m);
the moving average gives RM {fcst[fc.MA]:,.1f} m.

POSTER STRIP (centre panel)
  Method A  F(t) = (A(t-1)+A(t-2)+A(t-3)+A(t-4))/4   n = 4
  Method B  Y = a + bt   a = {a_all:,.2f}, b = {b_all:,.2f} (fitted on {len(df)} qtrs)
  Worked:   {target}: MA = RM {ma_f:,.1f} m | LS = RM {ls_f:,.1f} m | Actual = RM {actual:,.1f} m
  Accuracy: see Chart M3-3 (MAD / MSE / MAPE, identical quarters)
  Choice:   {SELECTED}: lower MSE on validation; best on all measures over 14 quarters; low bias

VIDEO NOTES - Member 3 (3:15-5:30)
  1. Why two methods: MA = stable baseline; LS = captures MR D.I.Y.'s growth trend.
  2. Fair test: rolling one-quarter-ahead forecasts, no future data, the same quarters for both methods.
  3. Walk through the {target} worked example on screen (Step 5 output).
  4. Show Chart M3-1 and M3-3: mixed results on 4 quarters, LS clearly better on 14.
  5. Decision: {SELECTED}; MA lags growth (bias +{ma_r['Bias (mean error)']:,.0f} m); hand-off to Streamlit.
'''
print(text)
with open("outputs/MRDIY_M3_Report_Text.txt", "w", encoding="utf-8") as fh:
    fh.write(text)

## Step 10 — Package the hand-off for Member 5 (Streamlit)
**How Member 5 uses it:** copy `src/` into the app folder, then:
```python
from src.forecasting import rolling_one_step_forecasts, forecast_next, apply_scenario, MA, LS
from src.accuracy import accuracy_table, select_method
```

In [ ]:
# Step 10: Bundle everything and download
import zipfile
with zipfile.ZipFile("MRDIY_M3_Handoff.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("src", "outputs"):
        for name in sorted(os.listdir(folder)):
            if not name.startswith(".") and name != "__pycache__":
                z.write(os.path.join(folder, name))
    z.write(DATA_FILE, "data/revenue.csv")      # Member 2's data, as the Streamlit plan expects

with zipfile.ZipFile("MRDIY_M3_Handoff.zip") as z:
    print("MRDIY_M3_Handoff.zip contains:")
    for n in z.namelist():
        print("  ", n)

try:
    from google.colab import files
    files.download("MRDIY_M3_Handoff.zip")
except ImportError:
    print("\n(Not running in Colab - zip saved in the working folder.)")